# Grassroots Music Venue — Analytics Database
### IOT552U Business Organisation and Decision Making
**What this notebook does:**
1. Generates all synthetic data programmatically (rule-based)
2. Creates the SQLite database with the full 3NF-normalised schema
3. Imports all 10 tables (4,083 rows total)
4. Runs 3 analytical SQL queries to evidence business intelligence value
5. Produces data visualisations for the reporting section

Run each cell **in order** from top to bottom (Shift+Enter or the ▶ button).


## Step 1 — Install libraries and set up imports

In [ ]:
# Install Plotly for interactive visualisations
!pip install plotly --quiet

import sqlite3
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

print("All libraries loaded successfully.")
print("Plotly version:", __import__('plotly').__version__)


## Step 2 — Generate synthetic data (rule-based)
All data is generated programmatically using explicit business rules.
Values are calibrated against published industry sources:
- **Music Venue Trust (2023):** ticket prices £8–£30 at grassroots venues
- **Musicians' Union:** artist fees £800–£2,500 for small venues  
- **Mailchimp Benchmarks (2024):** email open rate ~26%, click ~2.9%
- **CGA by NIQ (2024):** pint £4.50–£6.50, cocktail £8–£12
- **UKHospitality (2023):** avg per-head bar spend £10–£20


In [ ]:
"""
Grassroots Music Venue Analytics — Rule-Based Synthetic Data Generator
=======================================================================
Generates synthetic data for all 10 database tables using rule-based
programmatic generation (numpy + pandas).

Each generator function applies explicit business rules to derive values,
rather than drawing from unconstrained random distributions. This mirrors
real-world operational logic and ensures internal consistency across tables.

Calibration sources:
  - Music Venue Trust (2023): ticket prices £8–£30 at grassroots venues
  - Musicians' Union: artist fees £800–£2,500 for small-capacity venues
  - Mailchimp Benchmarks (2024): email open rate ~26%, click rate ~2.9%
  - CGA by NIQ (2024): pint £4.50–£6.50, cocktail £8–£12
  - UKHospitality (2023): avg per-head bar spend £10–£20

Run in Google Colab or locally. All CSVs are written to the working directory.
"""

import numpy as np
import pandas as pd

SEED = 42
rng  = np.random.default_rng(SEED)   # single seeded generator for reproducibility


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 1 — VenueRoom
# Fixed lookup table — two physical rooms, each with a fixed capacity.
# Capacity is a property of the room, not the event (3NF design decision).
# ══════════════════════════════════════════════════════════════════════════════

def generate_venue_rooms():
    data = [
        {"room_id": "RMM-001", "venue_room": "Main Hall", "capacity": 400},
        {"room_id": "RMM-002", "venue_room": "Lounge",    "capacity": 200},
    ]
    return pd.DataFrame(data)


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 2 — Promoter
# Fixed lookup — real-sounding independent promoter companies.
# ══════════════════════════════════════════════════════════════════════════════

def generate_promoters():
    names = [
        "Local Gig Co",     "Pulse Promotions",   "SummerSounds Ltd",
        "Blue Note Events", "Afro Vibes Co",       "Arts Council Presents",
        "StreetBeat Promo", "Indie Nights",        "Groove Factory",
        "Neon World Tours", "Roots and Routes",
    ]
    data = {
        "promoter_id":   [f"PRO-{str(i+1).zfill(3)}" for i in range(len(names))],
        "promoter_name": names,
    }
    return pd.DataFrame(data)


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 3 — Artist
#
# Rule: typical_fee is a base rate varied by ±10% to simulate negotiation.
# Fees calibrated to Musicians' Union grassroots rates (£800–£2,500).
# ══════════════════════════════════════════════════════════════════════════════

def generate_artists():
    base_data = [
        ("The Velvet Echoes",  "Rock",       "Manchester, UK",  1800),
        ("DJ Lumina",          "Electronic", "London, UK",      2200),
        ("Sara Montague",      "Jazz",       "London, UK",      1500),
        ("Kojo Beats",         "Afrobeats",  "London, UK",      2000),
        ("Midnight Strings",   "Classical",  "Birmingham, UK",  1200),
        ("Blaze MC",           "Hip-Hop",    "London, UK",      2500),
        ("Luna Park",          "Indie Pop",  "Bristol, UK",     1400),
        ("The Brass Collective","Funk",      "Manchester, UK",  1100),
        ("Anya Petrova",       "Folk",       "Edinburgh, UK",    800),
        ("Neon Pulse",         "Synthwave",  "Leeds, UK",       1600),
    ]
    n = len(base_data)

    # Rule: fee varies ±10% around base rate to simulate negotiation variance
    fee_multipliers = rng.uniform(0.90, 1.10, size=n)

    df = pd.DataFrame(base_data, columns=["artist_name","genre","origin","base_fee"])
    df["artist_id"]   = [f"ART-{str(i+1).zfill(3)}" for i in range(n)]
    df["typical_fee"] = (df["base_fee"] * fee_multipliers).round(2)
    return df[["artist_id","artist_name","genre","origin","typical_fee"]]


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 4 — VenueEvent
#
# Rules:
#   - Operating costs scale with room capacity (larger room = higher costs)
#   - Cost variation ±5% simulates real-world fluctuation (overtime, extra staff)
#   - Ticket prices follow MVT (2023) range: £8–£30 standard, higher for specials
# ══════════════════════════════════════════════════════════════════════════════

def generate_venue_events():
    templates = [
        # (date,         type,        genre,       room,      price, promoter,    staff, sec,  overhead)
        ("2025-06-14", "Concert",    "Rock",       "RMM-001", 18.00, "PRO-001",   420,   280,  600),
        ("2025-06-21", "Club Night", "Electronic", "RMM-001", 14.00, "PRO-002",   380,   300,  600),
        ("2025-07-05", "Concert",    "Mixed",      "RMM-001", 22.00, "PRO-003",   450,   310,  650),
        ("2025-07-12", "Concert",    "Jazz",       "RMM-002", 16.00, "PRO-004",   280,   180,  350),
        ("2025-07-26", "Concert",    "Afrobeats",  "RMM-001", 15.00, "PRO-005",   400,   270,  600),
        ("2025-08-09", "Concert",    "Classical",  "RMM-002", 22.00, "PRO-006",   300,   190,  400),
        ("2025-08-16", "Club Night", "Hip-Hop",    "RMM-001", 16.00, "PRO-007",   420,   320,  620),
        ("2025-09-06", "Concert",    "Mixed",      "RMM-001", 20.00, "PRO-003",   480,   330,  680),
        ("2025-09-13", "Concert",    "Indie Pop",  "RMM-002", 12.00, "PRO-008",   290,   180,  370),
        ("2025-09-27", "Concert",    "Funk",       "RMM-002", 10.00, "PRO-009",   270,   170,  340),
        ("2025-10-11", "Club Night", "Synthwave",  "RMM-001", 17.00, "PRO-010",   390,   270,  580),
        ("2025-10-25", "Concert",    "Folk",       "RMM-002", 10.00, "PRO-011",   240,   150,  300),
    ]
    n = len(templates)

    df = pd.DataFrame(templates, columns=[
        "event_date","event_type","genre","room_id",
        "base_ticket_price","promoter_id",
        "staffing_base","security_base","overhead_base"
    ])

    df["event_id"] = [f"EVT-{str(i+1).zfill(3)}" for i in range(n)]

    # Rule: costs vary ±5% around base to simulate real operational variance
    cost_variation = rng.uniform(0.95, 1.05, size=(n, 3))
    df["staffing_cost"]  = (df["staffing_base"]  * cost_variation[:, 0]).round(2)
    df["security_cost"]  = (df["security_base"]  * cost_variation[:, 1]).round(2)
    df["venue_overhead"] = (df["overhead_base"]  * cost_variation[:, 2]).round(2)

    # Start times: Club Nights start later (21:00–22:00), Concerts earlier (18:00–20:00)
    start_hours = np.where(df["event_type"] == "Club Night",
                           rng.integers(21, 23, size=n),
                           rng.integers(18, 21, size=n))
    start_mins  = rng.choice([0, 15, 30], size=n)
    df["start_time"] = [f"{h:02d}:{m:02d}:00" for h, m in zip(start_hours, start_mins)]

    cols = ["event_id","event_date","start_time","event_type","genre",
            "room_id","base_ticket_price","promoter_id",
            "staffing_cost","security_cost","venue_overhead"]
    return df[cols]


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 5 — EventArtist  (junction table resolving Artist ↔ VenueEvent M:N)
#
# Rules:
#   - Headliners (last in set order) paid 95–105% of typical_fee
#   - Support acts paid 80–95% (common industry practice)
#   - set_order 1 = opener, highest = headliner
# ══════════════════════════════════════════════════════════════════════════════

def generate_event_artists(artists_df):
    bookings = [
        ("EVT-001", ["ART-001"]),
        ("EVT-002", ["ART-002"]),
        ("EVT-003", ["ART-001","ART-004","ART-008","ART-007"]),
        ("EVT-004", ["ART-003"]),
        ("EVT-005", ["ART-004"]),
        ("EVT-006", ["ART-005"]),
        ("EVT-007", ["ART-006"]),
        ("EVT-008", ["ART-002","ART-006","ART-010","ART-009"]),
        ("EVT-009", ["ART-007"]),
        ("EVT-010", ["ART-008"]),
        ("EVT-011", ["ART-010"]),
        ("EVT-012", ["ART-009"]),
    ]

    # Build lookup: artist_id -> typical_fee
    fee_lookup = artists_df.set_index("artist_id")["typical_fee"].to_dict()

    rows = []
    ea_id = 1
    for event_id, artist_ids in bookings:
        n_acts = len(artist_ids)
        for set_order, artist_id in enumerate(artist_ids, start=1):
            typical = fee_lookup[artist_id]
            is_headliner = (set_order == n_acts)

            # Rule: headliner fee closer to typical; support acts negotiated down
            if is_headliner:
                multiplier = rng.uniform(0.95, 1.05)
            else:
                multiplier = rng.uniform(0.80, 0.95)

            rows.append({
                "event_artist_id": f"EA-{str(ea_id).zfill(3)}",
                "event_id":        event_id,
                "artist_id":       artist_id,
                "artist_fee_paid": round(float(typical) * multiplier, 2),
                "set_order":       set_order,
            })
            ea_id += 1

    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 6 — Customer  (60 anonymised profiles)
#
# Rules:
#   - Age band weights reflect typical grassroots venue demographics
#     (younger-skewing audience, 18–34 dominant)
#   - accessibility_needs assigned with realistic low probability (~11%)
#   - is_student and customer_segment removed — derivable from age_band (3NF fix)
# ══════════════════════════════════════════════════════════════════════════════

def generate_customers(n=60):
    age_bands   = ["18-24", "25-34", "35-44", "45-54", "55-64"]
    age_weights = np.array([0.30,   0.35,   0.20,   0.10,   0.05])

    postcodes = [
        "SW1 1","EC2 4","M1 2","B1 1","LS1 5","EH1 3","CF10 1","SW3 2",
        "E1 6","M4 4","BS1 3","SE1 7","N1 0","NG1 5","L1 8","G1 2",
        "SW11 1","OX1 2","E14 5","M20 3","W1 4","CB1 1","SE16 3","LE1 6",
        "SW6 2","BA1 1","EC1 8","NE1 4","WC2 5","SE11 4","M13 9","SW19 2",
        "E2 8","LS2 7","N7 6","EH3 9","W6 0","BS2 0","L3 4","SE5 7",
        "NG7 2","CF11 6","G2 1","OX2 6","M15 5","SW7 3","EC3 7","CB2 3",
        "SE10 8","W4 1","NE4 5","LE2 7","WC1 6","BA2 4","N16 8","SW15 1",
        "M1 5","E3 4","BS8 1","L2 2",
    ]
    access_options = ["None"] * 9 + ["Wheelchair access", "Hearing loop"]

    # Rule: age band sampled according to demographic weights
    chosen_ages = rng.choice(age_bands, size=n, p=age_weights / age_weights.sum())

    df = pd.DataFrame({
        "customer_id":         [f"CST-{str(i+1).zfill(3)}" for i in range(n)],
        "age_band":            chosen_ages,
        "postcode_sector":     [postcodes[i % len(postcodes)] for i in range(n)],
        "accessibility_needs": rng.choice(access_options, size=n),
    })
    return df


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 7 — PromotionCampaign
#
# Rules:
#   - Email open rates: Mailchimp Arts & Entertainment benchmark ~26% mean
#   - open_rate is NULL for non-email channels (not applicable — stored as NaN)
#   - Budget capped at £500 (realistic grassroots marketing spend)
#   - Campaign window ends 2 days before event; starts 18–25 days before
# ══════════════════════════════════════════════════════════════════════════════

def generate_campaigns(events_df):
    templates = [
        ("EVT-001", "Email",       "ROCK15",    15.0),
        ("EVT-002", "Social Media","ELECTRO10", 10.0),
        ("EVT-003", "Email",       "MIXED20",   20.0),
        ("EVT-005", "Social Media","AFRO10",    10.0),
        ("EVT-007", "Influencer",  "HIPHOP15",  15.0),
        ("EVT-008", "Email",       "AUTUMN20",  20.0),
        ("EVT-011", "Social Media","NEON10",    10.0),
    ]
    n = len(templates)

    event_dates = events_df.set_index("event_id")["event_date"].to_dict()

    rows = []
    for i, (event_id, channel, code, disc_pct) in enumerate(templates):
        ev_date   = pd.to_datetime(event_dates[event_id])
        end_date  = (ev_date - pd.Timedelta(days=2)).strftime("%Y-%m-%d")
        days_back = int(rng.integers(18, 26))
        start_date = (ev_date - pd.Timedelta(days=days_back)).strftime("%Y-%m-%d")

        # Rule: open_rate only applies to Email channel (Mailchimp benchmark ~26%)
        if channel == "Email":
            open_rate = round(float(rng.normal(loc=29.0, scale=3.0)), 2)
        else:
            open_rate = np.nan   # NULL in database — not applicable

        # Rule: click rate higher for Influencer (higher engagement) than Email
        if channel == "Influencer":
            click_rate = round(float(rng.uniform(12.0, 18.0)), 2)
        else:
            click_rate = round(float(rng.uniform(7.0, 14.0)), 2)

        conversion_rate = round(float(rng.uniform(3.0, 7.5)), 2)

        rows.append({
            "campaign_id":         f"CMP-{str(i+1).zfill(3)}",
            "event_id":            event_id,
            "campaign_channel":    channel,
            "discount_code":       code,
            "discount_percentage": disc_pct,
            "start_date":          start_date,
            "end_date":            end_date,
            "budget":              round(float(rng.uniform(200.0, 500.0)), 2),
            "open_rate":           open_rate,
            "click_rate":          click_rate,
            "conversion_rate":     conversion_rate,
        })

    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 8 — TicketSale  (2,500–3,000 rows expected)
#
# Rules (all explicitly coded):
#   1. Sell-through rate varies by genre (Electronic 90%, Folk 55%)
#   2. 18–22% of tickets use a campaign discount → price = base × (1 − disc%)
#   3. 7% are VIP → price = base × 1.4
#   4. No-show rate: 12–15% of sold tickets (attended = 0, refunded = 0)
#   5. Refund rate: 4–5% of sold tickets (attended = 0, refunded = 1)
#   6. Booking channel: 70% Online, 15% Box Office, 15% Door
#   7. Customer selection biased by genre (younger for Electronic, older for Jazz)
#   8. Sale datetime spread over 21 days before event; door sales on event day
# ══════════════════════════════════════════════════════════════════════════════

# Genre → target sell-through rate (proportion of room capacity sold)
SELL_THROUGH = {
    "Rock": 0.85, "Electronic": 0.90, "Mixed": 0.75, "Jazz": 0.60,
    "Afrobeats": 0.80, "Classical": 0.65, "Hip-Hop": 0.88,
    "Indie Pop": 0.65, "Funk": 0.58, "Synthwave": 0.82, "Folk": 0.55,
}

# Genre → customer age-band sampling weights [18-24, 25-34, 35-44, 45-54, 55-64]
GENRE_AGE_WEIGHTS = {
    "Electronic": [0.40, 0.40, 0.15, 0.04, 0.01],
    "Hip-Hop":    [0.45, 0.35, 0.15, 0.04, 0.01],
    "Synthwave":  [0.35, 0.40, 0.18, 0.06, 0.01],
    "Rock":       [0.30, 0.35, 0.22, 0.10, 0.03],
    "Afrobeats":  [0.30, 0.40, 0.20, 0.08, 0.02],
    "Indie Pop":  [0.35, 0.38, 0.18, 0.07, 0.02],
    "Mixed":      [0.28, 0.35, 0.22, 0.10, 0.05],
    "Funk":       [0.20, 0.35, 0.25, 0.15, 0.05],
    "Jazz":       [0.10, 0.20, 0.30, 0.25, 0.15],
    "Classical":  [0.08, 0.15, 0.28, 0.30, 0.19],
    "Folk":       [0.10, 0.20, 0.30, 0.25, 0.15],
}

AGE_BANDS = ["18-24", "25-34", "35-44", "45-54", "55-64"]

def generate_ticket_sales(events_df, customers_df, campaigns_df, rooms_df):
    # Build lookup dictionaries from DataFrames
    room_cap     = rooms_df.set_index("room_id")["capacity"].to_dict()
    camp_by_ev   = campaigns_df.set_index("event_id").to_dict("index")

    # Group customer IDs by age band for genre-biased selection
    custs_by_age = customers_df.groupby("age_band")["customer_id"].apply(list).to_dict()

    all_rows      = []
    ticket_id     = 1
    attendees_map = {}   # event_id -> [customer_ids who attended]

    for _, event in events_df.iterrows():
        event_id   = event["event_id"]
        genre      = event["genre"]
        event_date = event["event_date"]
        base_price = float(event["base_ticket_price"])
        capacity   = room_cap[event["room_id"]]
        vip_price  = round(base_price * 1.4, 2)

        # Rule 1: sell-through determines how many tickets to generate
        n_tickets  = int(capacity * SELL_THROUGH.get(genre, 0.70))

        # Rule 2 & 3: determine ticket type composition
        campaign   = camp_by_ev.get(event_id)
        disc_pct   = float(campaign["discount_percentage"]) if campaign else 0
        disc_price = round(base_price * (1 - disc_pct / 100), 2) if campaign else None
        camp_id    = campaign["campaign_id"] if campaign else None

        n_discount = int(n_tickets * rng.uniform(0.18, 0.22)) if campaign else 0
        n_vip      = int(n_tickets * rng.uniform(0.06, 0.09))
        n_standard = n_tickets - n_discount - n_vip

        ticket_types = (
            [("Early Bird", disc_price, camp_id)] * n_discount +
            [("VIP",        vip_price,  None)]    * n_vip +
            [("Standard",   base_price, None)]    * n_standard
        )
        rng.shuffle(ticket_types)

        # Rule 4 & 5: attendance outcomes
        no_show_rate = rng.uniform(0.12, 0.15)
        refund_rate  = rng.uniform(0.04, 0.05)
        outcomes     = rng.random(size=n_tickets)   # one draw per ticket

        # Rule 7: genre-biased customer pool
        age_weights = np.array(GENRE_AGE_WEIGHTS.get(genre, [0.2]*5))
        age_weights = age_weights / age_weights.sum()

        # Rule 8: sale date spread
        ev_dt    = pd.to_datetime(event_date)
        sale_days = rng.integers(0, 21, size=n_tickets)  # days before event

        attendees_map[event_id] = []

        for j, (ttype, price, used_camp_id) in enumerate(ticket_types):
            # Genre-biased customer selection
            age_band    = rng.choice(AGE_BANDS, p=age_weights)
            pool        = custs_by_age.get(age_band, customers_df["customer_id"].tolist())
            customer_id = str(rng.choice(pool))

            # Rule 6: booking channel
            channel = rng.choice(
                ["Online", "Box Office", "Door"],
                p=[0.70, 0.15, 0.15]
            )

            # Rule 8: door sales happen on event day; pre-sales spread over 21 days
            if channel == "Door":
                sale_dt = f"{event_date} {rng.integers(17,20):02d}:{rng.integers(0,59):02d}:00"
            else:
                sale_date = (ev_dt - pd.Timedelta(days=int(sale_days[j]))).strftime("%Y-%m-%d")
                sale_dt   = f"{sale_date} {rng.integers(8,21):02d}:{rng.integers(0,59):02d}:{rng.integers(0,59):02d}"

            # Rules 4 & 5: attendance status derived from random draw
            o = outcomes[j]
            if o < refund_rate:
                attended = 0; refunded = 1
            elif o < refund_rate + no_show_rate:
                attended = 0; refunded = 0
            else:
                attended = 1; refunded = 0

            if attended:
                attendees_map[event_id].append(customer_id)

            all_rows.append({
                "ticket_sale_id":  f"TKT-{str(ticket_id).zfill(4)}",
                "event_id":        event_id,
                "customer_id":     customer_id,
                "campaign_id":     used_camp_id if used_camp_id else "",
                "sale_datetime":   sale_dt,
                "price_paid":      price,
                "ticket_type":     ttype,
                "booking_channel": channel,
                "attended":        attended,
                "refunded":        refunded,
            })
            ticket_id += 1

    return pd.DataFrame(all_rows), attendees_map


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 9 — BarOrder
#
# Rules:
#   1. ONLY customers with attended=True can have a bar order
#      (enforced by reading from attendees_map built in TicketSale generation)
#   2. Bar order rate varies by genre (Electronic 78%, Folk 60%)
#   3. Spend drawn from genre-calibrated range (CGA by NIQ / UKHospitality)
#   4. 20% of orders anonymous (customer_id NULL — nullable FK design decision)
#   5. Timestamp: 30–150 minutes after event start
# ══════════════════════════════════════════════════════════════════════════════

# Genre → (min_spend, max_spend, order_probability)
GENRE_BAR = {
    "Rock":       (8.0,  28.0, 0.72),
    "Electronic": (10.0, 35.0, 0.78),
    "Mixed":      (9.0,  30.0, 0.74),
    "Jazz":       (12.0, 30.0, 0.68),
    "Afrobeats":  (9.0,  28.0, 0.70),
    "Classical":  (10.0, 28.0, 0.62),
    "Hip-Hop":    (10.0, 32.0, 0.76),
    "Indie Pop":  (7.0,  22.0, 0.65),
    "Funk":       (8.0,  24.0, 0.68),
    "Synthwave":  (10.0, 34.0, 0.77),
    "Folk":       (8.0,  20.0, 0.60),
}

def generate_bar_orders(events_df, attendees_map):
    all_rows = []
    order_id = 1

    for _, event in events_df.iterrows():
        event_id   = event["event_id"]
        genre      = event["genre"]
        event_date = event["event_date"]
        min_spend, max_spend, bar_rate = GENRE_BAR.get(genre, (8.0, 25.0, 0.70))

        # Rule 1: only use attendees confirmed from ticket sales
        attendees = list(set(attendees_map.get(event_id, [])))
        n = len(attendees)
        if n == 0:
            continue

        # Rule 2: apply bar visit probability per attendee
        visits = rng.random(size=n) < bar_rate

        # Rule 3: spend drawn uniformly within genre range
        spends = np.round(rng.uniform(min_spend, max_spend, size=n) * 2) / 2

        # Rule 5: order timestamps 30–150 minutes after doors (start_time)
        start_hour = int(event["start_time"].split(":")[0])
        minutes_in = rng.integers(30, 151, size=n)
        order_hours = start_hour + minutes_in // 60
        order_mins  = minutes_in % 60

        # Rule 4: 20% anonymous orders
        anonymous_mask = rng.random(size=n) < 0.20
        channels       = rng.choice(["Bar", "App"], size=n, p=[0.80, 0.20])

        for j, customer_id in enumerate(attendees):
            if not visits[j]:
                continue   # customer didn't visit the bar tonight

            linked_customer = "" if anonymous_mask[j] else customer_id

            all_rows.append({
                "bar_order_id":    f"BAR-{str(order_id).zfill(3)}",
                "event_id":        event_id,
                "customer_id":     linked_customer,
                "order_timestamp": f"{event_date} {int(order_hours[j]):02d}:{int(order_mins[j]):02d}:00",
                "total_spend":     spends[j],
                "order_channel":   channels[j],
            })
            order_id += 1

    return pd.DataFrame(all_rows)


# ══════════════════════════════════════════════════════════════════════════════
# TABLE 10 — BarOrderItem  (1NF fix — atomic line items per bar order)
#
# Rule: each order is decomposed into 1–3 distinct drink types with quantity.
# Drink type mix weighted by realistic on-trade sales (beer most common).
# ══════════════════════════════════════════════════════════════════════════════

DRINK_TYPES   = ["Beer", "Wine", "Cocktail", "Soft Drink"]
DRINK_WEIGHTS = np.array([0.45,  0.25,  0.20,    0.10])

def generate_bar_order_items(bar_orders_df):
    all_rows = []
    item_id  = 1
    n_orders = len(bar_orders_df)

    # Rule: number of distinct item types per order (1 most common)
    n_items_per_order = rng.choice([1, 2, 3], size=n_orders, p=[0.55, 0.35, 0.10])
    quantities        = rng.choice([1, 2, 3], size=(n_orders, 3), p=[0.55, 0.35, 0.10])

    for idx, (_, order) in enumerate(bar_orders_df.iterrows()):
        n_types = n_items_per_order[idx]
        chosen_drinks = rng.choice(DRINK_TYPES, size=n_types,
                                   p=DRINK_WEIGHTS / DRINK_WEIGHTS.sum(),
                                   replace=False)
        for k, drink in enumerate(chosen_drinks):
            all_rows.append({
                "order_item_id": f"ITM-{str(item_id).zfill(4)}",
                "bar_order_id":  order["bar_order_id"],
                "item_name":     drink,
                "quantity":      int(quantities[idx, k]),
            })
            item_id += 1

    return pd.DataFrame(all_rows)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN — run all generators in FK-dependency order and export to CSV
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("\nGrassroots Venue Data Generator (Rule-Based)")
    print("=" * 52)

    venue_rooms  = generate_venue_rooms()
    promoters    = generate_promoters()
    artists      = generate_artists()
    events       = generate_venue_events()
    event_artists = generate_event_artists(artists)
    customers    = generate_customers(n=60)
    campaigns    = generate_campaigns(events)

    tickets, attendees_map = generate_ticket_sales(
        events, customers, campaigns, venue_rooms
    )

    bar_orders   = generate_bar_orders(events, attendees_map)
    bar_items    = generate_bar_order_items(bar_orders)

    # Export all tables to CSV
    tables = {
        "venue_room.csv":         venue_rooms,
        "promoter.csv":           promoters,
        "artist.csv":             artists,
        "venue_event.csv":        events,
        "event_artist.csv":       event_artists,
        "customer.csv":           customers,
        "promotion_campaign.csv": campaigns,
        "ticket_sale.csv":        tickets,
        "bar_order.csv":          bar_orders,
        "bar_order_item.csv":     bar_items,
    }

    for filename, df in tables.items():
        df.to_csv(filename, index=False)
        print(f"  {filename:40s}  {len(df):>5,} rows")

    # Summary statistics
    print("\nSummary Statistics")
    print("-" * 52)
    attended = tickets["attended"].sum()
    refunded = tickets["refunded"].sum()
    print(f"  Total ticket sales:   {len(tickets):>5,}")
    print(f"    Attended:           {attended:>5,}  ({attended/len(tickets)*100:.1f}%)")
    print(f"    Refunded:           {refunded:>5,}  ({refunded/len(tickets)*100:.1f}%)")
    print(f"  Bar orders:           {len(bar_orders):>5,}")
    print(f"  Bar order items:      {len(bar_items):>5,}")

    print("\nSell-through by event:")
    room_cap = venue_rooms.set_index("room_id")["capacity"].to_dict()
    ticket_counts = tickets.groupby("event_id").size()
    for _, ev in events.iterrows():
        cap  = room_cap[ev["room_id"]]
        sold = ticket_counts.get(ev["event_id"], 0)
        print(f"  {ev['event_id']}  {ev['genre']:<12}  {sold:>3}/{cap}  ({sold/cap*100:.0f}%)")

    print("\nAll CSVs generated.")

    # Preview first table
    print("\nTicketSale preview (first 5 rows):")
    print(tickets.head())


## Step 3 — Create the SQLite database
The schema below was designed through normalisation analysis (1NF → 2NF → 3NF → BCNF).
See the Data Model section of the report for full justification of design decisions.


In [ ]:
# ── Schema definition ────────────────────────────────────────────────────
# Each CREATE TABLE statement reflects the normalised data model.
# Tables are created in FK-dependency order (parent before child).

SCHEMA_SQL = """
-- ============================================================
-- Grassroots Music Venue Analytics Database
-- Schema Definition (3NF Normalised)
-- ============================================================


-- 1. VenueRoom
--    Stores the physical rooms in the venue.
--    Extracted from VenueEvent to remove the transitive dependency
--    venue_room → capacity (3NF fix).
-- ------------------------------------------------------------
CREATE TABLE VenueRoom (
    room_id       VARCHAR(10)  PRIMARY KEY,
    venue_room    VARCHAR(50)  NOT NULL,
    capacity      INT          NOT NULL
);


-- 2. Promoter
--    Stores promoter details as a separate entity.
--    Extracted from VenueEvent to remove the transitive dependency
--    promoter_name → (any future promoter attributes) (3NF fix).
-- ------------------------------------------------------------
CREATE TABLE Promoter (
    promoter_id   VARCHAR(10)  PRIMARY KEY,
    promoter_name VARCHAR(100) NOT NULL
);


-- 3. Artist
--    Stores artist profile information.
--    typical_fee is the artist's standard asking price,
--    separate from what was actually paid per event (held in EventArtist).
-- ------------------------------------------------------------
CREATE TABLE Artist (
    artist_id    VARCHAR(10)  PRIMARY KEY,
    artist_name  VARCHAR(100) NOT NULL,
    genre        VARCHAR(50),
    origin       VARCHAR(100),
    typical_fee  DECIMAL(8,2)
);


-- 4. VenueEvent
--    Stores each individual event.
--    room_id and promoter_id are foreign keys replacing the
--    raw text columns that caused 3NF violations.
--    capacity is no longer stored here — it belongs to VenueRoom.
-- ------------------------------------------------------------
CREATE TABLE VenueEvent (
    event_id          VARCHAR(10)  PRIMARY KEY,
    event_date        DATE         NOT NULL,
    start_time        TIME,
    event_type        VARCHAR(50),
    genre             VARCHAR(50),
    room_id           VARCHAR(10)  NOT NULL,
    base_ticket_price DECIMAL(6,2),
    promoter_id       VARCHAR(10)  NOT NULL,
    staffing_cost     DECIMAL(8,2),
    security_cost     DECIMAL(8,2),
    venue_overhead    DECIMAL(8,2),
    FOREIGN KEY (room_id)     REFERENCES VenueRoom(room_id),
    FOREIGN KEY (promoter_id) REFERENCES Promoter(promoter_id)
);


-- 5. EventArtist
--    Junction table linking artists to events (many-to-many).
--    artist_fee_paid records what was actually paid on the night,
--    which may differ from the artist's typical_fee.
--    set_order records the running order of acts (1 = opener, highest = headliner).
-- ------------------------------------------------------------
CREATE TABLE EventArtist (
    event_artist_id VARCHAR(10)  PRIMARY KEY,
    event_id        VARCHAR(10)  NOT NULL,
    artist_id       VARCHAR(10)  NOT NULL,
    artist_fee_paid DECIMAL(8,2),
    set_order       INT,
    FOREIGN KEY (event_id)  REFERENCES VenueEvent(event_id),
    FOREIGN KEY (artist_id) REFERENCES Artist(artist_id)
);


-- 6. Customer
--    Stores anonymised customer profile data.
--    is_student and customer_segment have been removed as both
--    were entirely derivable from age_band — storing them would
--    create redundancy and risk contradictory data (2NF/3NF fix).
-- ------------------------------------------------------------
CREATE TABLE Customer (
    customer_id        VARCHAR(10)  PRIMARY KEY,
    age_band           VARCHAR(20),
    postcode_sector    VARCHAR(10),
    accessibility_needs VARCHAR(100)
);


-- 7. PromotionCampaign
--    Stores marketing campaigns linked to specific events.
--    open_rate is NULL for non-email channels (Social Media,
--    Influencer) as that metric does not apply.
-- ------------------------------------------------------------
CREATE TABLE PromotionCampaign (
    campaign_id         VARCHAR(10)  PRIMARY KEY,
    event_id            VARCHAR(10)  NOT NULL,
    campaign_channel    VARCHAR(50),
    discount_code       VARCHAR(30),
    discount_percentage DECIMAL(4,2),
    start_date          DATE,
    end_date            DATE,
    budget              DECIMAL(8,2),
    open_rate           DECIMAL(5,2),
    click_rate          DECIMAL(5,2),
    conversion_rate     DECIMAL(5,2),
    FOREIGN KEY (event_id) REFERENCES VenueEvent(event_id)
);


-- 8. TicketSale
--    Records every individual ticket purchase.
--    campaign_id is nullable — most tickets are sold without a promotion.
--    attended and refunded default to FALSE.
-- ------------------------------------------------------------
CREATE TABLE TicketSale (
    ticket_sale_id  VARCHAR(10)  PRIMARY KEY,
    event_id        VARCHAR(10)  NOT NULL,
    customer_id     VARCHAR(10)  NOT NULL,
    campaign_id     VARCHAR(10),
    sale_datetime   DATETIME     NOT NULL,
    price_paid      DECIMAL(6,2) NOT NULL,
    ticket_type     VARCHAR(30),
    booking_channel VARCHAR(30),
    attended        BOOLEAN      DEFAULT FALSE,
    refunded        BOOLEAN      DEFAULT FALSE,
    FOREIGN KEY (event_id)    REFERENCES VenueEvent(event_id),
    FOREIGN KEY (customer_id) REFERENCES Customer(customer_id),
    FOREIGN KEY (campaign_id) REFERENCES PromotionCampaign(campaign_id)
);


-- 9. BarOrder
--    Records each bar transaction during an event.
--    customer_id is nullable to allow anonymous walk-up purchases.
--    The items column has been removed — individual items are now
--    stored atomically in BarOrderItem (1NF fix).
-- ------------------------------------------------------------
CREATE TABLE BarOrder (
    bar_order_id    VARCHAR(10)  PRIMARY KEY,
    event_id        VARCHAR(10)  NOT NULL,
    customer_id     VARCHAR(10),
    order_timestamp DATETIME     NOT NULL,
    total_spend     DECIMAL(6,2) NOT NULL,
    order_channel   VARCHAR(30),
    FOREIGN KEY (event_id)    REFERENCES VenueEvent(event_id),
    FOREIGN KEY (customer_id) REFERENCES Customer(customer_id)
);


-- 10. BarOrderItem
--     Stores individual drink items per bar order atomically.
--     Extracted from BarOrder to resolve the 1NF violation where
--     multiple items were stored as a single text string (e.g. "2x Beer 1x Wine").
--     Each row now represents exactly one item type and quantity.
-- ------------------------------------------------------------
CREATE TABLE BarOrderItem (
    order_item_id VARCHAR(10)  PRIMARY KEY,
    bar_order_id  VARCHAR(10)  NOT NULL,
    item_name     VARCHAR(50)  NOT NULL,
    quantity      INT          NOT NULL,
    FOREIGN KEY (bar_order_id) REFERENCES BarOrder(bar_order_id)
);

"""

# Create (or overwrite) the database file
DB_PATH = "GrassrootsVenue.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)

# Disable FK enforcement during bulk import for performance,
# then re-enable for all subsequent queries
conn.execute("PRAGMA foreign_keys = OFF")

# Run each CREATE TABLE statement individually
import re
statements = [s.strip() for s in SCHEMA_SQL.split(';')
              if s.strip() and 'CREATE TABLE' in s]

for stmt in statements:
    conn.execute(stmt)
    table_name = re.search(r'CREATE TABLE (\w+)', stmt).group(1)
    print(f"  Created table: {table_name}")

conn.commit()
print(f"\nDatabase created: {DB_PATH}")


## Step 4 — Import all generated data into the database
Data is loaded from the DataFrames generated in Step 2.
Tables are loaded in FK-safe order — parent tables before child tables.


In [ ]:
# ── Map DataFrame variable names to SQL table names ─────────────────────
TABLE_ORDER = [
    (venue_rooms,   "VenueRoom"),
    (promoters,     "Promoter"),
    (artists,       "Artist"),
    (events,        "VenueEvent"),
    (event_artists, "EventArtist"),
    (customers,     "Customer"),
    (campaigns,     "PromotionCampaign"),
    (tickets,       "TicketSale"),
    (bar_orders,    "BarOrder"),
    (bar_items,     "BarOrderItem"),
]

conn.execute("PRAGMA foreign_keys = OFF")

for df, table_name in TABLE_ORDER:
    # Replace NaN with None so sqlite3 stores NULL correctly
    clean_df = df.where(pd.notnull(df), None)
    cols     = list(clean_df.columns)
    col_str  = ", ".join(cols)
    placeholders = ", ".join(["?" for _ in cols])
    insert_sql   = f"INSERT INTO {table_name} ({col_str}) VALUES ({placeholders})"

    rows_inserted = 0
    for _, row in clean_df.iterrows():
        conn.execute(insert_sql, list(row))
        rows_inserted += 1

    conn.commit()
    count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"  {table_name:<25}  {count:>5,} rows")

# Re-enable FK enforcement for all queries going forward
conn.execute("PRAGMA foreign_keys = ON")
conn.commit()
print("\nAll tables imported. Foreign key enforcement enabled.")


## Step 5 — Verify the database

In [ ]:
# Quick sanity check — count rows and preview each table
tables = [
    "VenueRoom","Promoter","Artist","VenueEvent","EventArtist",
    "Customer","PromotionCampaign","TicketSale","BarOrder","BarOrderItem"
]

print(f"{'Table':<25} {'Rows':>6}")
print("-" * 33)
for t in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<23} {count:>6,}")

print("\nSample from TicketSale (first 5 rows):")
df_check = pd.read_sql("SELECT * FROM TicketSale LIMIT 5", conn)
print(df_check.to_string(index=False))


## Step 6 — Analytical SQL Queries

The three queries below each draw from **more than one table** using JOIN, 
directly answering the three research questions set out in the introduction.

Each cell prints:
1. The raw SQL so it is clearly visible
2. The results as a formatted table


### Query 1 — Event Revenue and Profitability
**Research question:** Which event types and artists generate the strongest attendance and revenue?

**Tables joined:** VenueEvent · VenueRoom · TicketSale · EventArtist


In [ ]:
# ── Query 1: Event Revenue and Profitability ─────────────────────────────
QUERY_1 = """
SELECT
    ve.event_id,
    ve.event_date,
    ve.genre,
    ve.event_type,
    vr.venue_room,
    vr.capacity,
    COUNT(ts.ticket_sale_id)                                AS tickets_sold,
    ROUND(COUNT(ts.ticket_sale_id) * 100.0
          / vr.capacity, 1)                                 AS sell_through_pct,
    SUM(CASE WHEN ts.attended = 1 THEN 1 ELSE 0 END)       AS actual_attendees,
    ROUND(SUM(ts.price_paid), 2)                            AS ticket_revenue,
    ROUND(SUM(ea.artist_fee_paid), 2)                       AS artist_fees,
    ROUND(ve.staffing_cost + ve.security_cost
          + ve.venue_overhead, 2)                           AS operating_costs,
    ROUND(SUM(ts.price_paid)
          - SUM(ea.artist_fee_paid)
          - ve.staffing_cost
          - ve.security_cost
          - ve.venue_overhead, 2)                           AS estimated_profit
FROM VenueEvent    ve
JOIN VenueRoom     vr ON ve.room_id  = vr.room_id
JOIN TicketSale    ts ON ve.event_id = ts.event_id
JOIN EventArtist   ea ON ve.event_id = ea.event_id
GROUP BY ve.event_id
ORDER BY ticket_revenue DESC
"""

# Print the SQL so it is clearly visible in the notebook output
print("=" * 60)
print("SQL QUERY 1 — Event Revenue and Profitability")
print("=" * 60)
print(QUERY_1)

# Run the query and display results
df_q1 = pd.read_sql(QUERY_1, conn)

print("RESULTS:")
print("-" * 60)
print(df_q1.to_string(index=False))
print(f"\n{len(df_q1)} rows returned.")


### Query 2 — Bar Spend vs Ticket Attendance by Genre
**Research question:** How do ticket sales relate to bar revenue and total event profitability?

**Tables joined:** VenueEvent · TicketSale · BarOrder


In [ ]:
# ── Query 2: Bar Spend vs Attendance by Genre ────────────────────────────
QUERY_2 = """
SELECT
    ve.genre,
    ve.event_type,
    COUNT(DISTINCT ts.ticket_sale_id)                       AS tickets_sold,
    SUM(CASE WHEN ts.attended = 1 THEN 1 ELSE 0 END)       AS attendees,
    COUNT(DISTINCT bo.bar_order_id)                         AS bar_orders,
    ROUND(SUM(bo.total_spend), 2)                           AS total_bar_revenue,
    ROUND(AVG(bo.total_spend), 2)                           AS avg_spend_per_order,
    ROUND(
        SUM(bo.total_spend)
        / NULLIF(SUM(CASE WHEN ts.attended = 1
                          THEN 1 ELSE 0 END), 0),
    2)                                                      AS bar_spend_per_attendee
FROM VenueEvent ve
JOIN TicketSale ts ON ve.event_id = ts.event_id
LEFT JOIN BarOrder bo ON ve.event_id = bo.event_id
GROUP BY ve.genre
ORDER BY bar_spend_per_attendee DESC
"""

print("=" * 60)
print("SQL QUERY 2 — Bar Spend vs Attendance by Genre")
print("=" * 60)
print(QUERY_2)

df_q2 = pd.read_sql(QUERY_2, conn)

print("RESULTS:")
print("-" * 60)
print(df_q2.to_string(index=False))
print(f"\n{len(df_q2)} rows returned.")


### Query 3 — Customer Segment Responsiveness to Promotions
**Research question:** Which customer segments are most responsive to different event offerings or pricing strategies?

**Tables joined:** Customer · TicketSale · PromotionCampaign


In [ ]:
# ── Query 3: Customer Segment Analysis ──────────────────────────────────
QUERY_3 = """
SELECT
    c.age_band,
    COUNT(ts.ticket_sale_id)                                AS total_tickets,
    SUM(CASE WHEN ts.campaign_id != ''
              AND ts.campaign_id IS NOT NULL
              THEN 1 ELSE 0 END)                            AS discounted_tickets,
    ROUND(
        SUM(CASE WHEN ts.campaign_id != ''
                  AND ts.campaign_id IS NOT NULL
                  THEN 1 ELSE 0 END) * 100.0
        / COUNT(ts.ticket_sale_id),
    1)                                                      AS discount_usage_pct,
    ROUND(AVG(ts.price_paid), 2)                            AS avg_price_paid,
    SUM(CASE WHEN ts.attended = 1 THEN 1 ELSE 0 END)       AS attended,
    ROUND(
        SUM(CASE WHEN ts.attended = 1 THEN 1 ELSE 0 END)
        * 100.0 / COUNT(ts.ticket_sale_id),
    1)                                                      AS attendance_rate_pct,
    SUM(CASE WHEN ts.ticket_type = 'VIP'
             THEN 1 ELSE 0 END)                             AS vip_tickets
FROM Customer   c
JOIN TicketSale ts ON c.customer_id = ts.customer_id
GROUP BY c.age_band
ORDER BY c.age_band
"""

print("=" * 60)
print("SQL QUERY 3 — Customer Segment Responsiveness to Promotions")
print("=" * 60)
print(QUERY_3)

df_q3 = pd.read_sql(QUERY_3, conn)

print("RESULTS:")
print("-" * 60)
print(df_q3.to_string(index=False))
print(f"\n{len(df_q3)} rows returned.")


## Step 7 — Interactive Visualisations (Plotly)

All three charts are **interactive** — hover over any element to see exact values,
click legend items to show/hide series, and zoom/pan using the toolbar.

Each visualisation is justified against the research questions and saved as HTML
(interactive) and PNG (for the report).


### Visualisation 1 — Ticket Revenue, Bar Revenue and Estimated Profit by Genre

**Justification:** A grouped bar chart is the most appropriate visualisation for 
comparing three related financial metrics across categorical groups (genres). 
It allows the venue manager to immediately identify which genres are profitable 
versus which generate high ticket revenue but low margins — directly supporting 
programming and pricing decisions (research question 1).


In [ ]:
# ── Visualisation 1: Revenue vs Profit by Genre (Plotly) ─────────────────
genre_finance = pd.read_sql("""
    SELECT
        ve.genre,
        ROUND(SUM(ts.price_paid), 2)                        AS ticket_revenue,
        ROUND(COALESCE(SUM(bo.total_spend), 0), 2)          AS bar_revenue,
        ROUND(SUM(ts.price_paid)
              + COALESCE(SUM(bo.total_spend), 0)
              - SUM(ea.artist_fee_paid)
              - ve.staffing_cost
              - ve.security_cost
              - ve.venue_overhead, 2)                       AS est_profit
    FROM VenueEvent  ve
    JOIN TicketSale  ts ON ve.event_id = ts.event_id
    JOIN EventArtist ea ON ve.event_id = ea.event_id
    LEFT JOIN BarOrder bo ON ve.event_id = bo.event_id
    GROUP BY ve.genre
    ORDER BY ticket_revenue DESC
""", conn)

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    name="Ticket Revenue",
    x=genre_finance["genre"],
    y=genre_finance["ticket_revenue"],
    marker_color="#2e4a7a",
    hovertemplate="<b>%{x}</b><br>Ticket Revenue: £%{y:,.2f}<extra></extra>"
))

fig1.add_trace(go.Bar(
    name="Bar Revenue",
    x=genre_finance["genre"],
    y=genre_finance["bar_revenue"],
    marker_color="#2e8b57",
    hovertemplate="<b>%{x}</b><br>Bar Revenue: £%{y:,.2f}<extra></extra>"
))

fig1.add_trace(go.Bar(
    name="Estimated Profit",
    x=genre_finance["genre"],
    y=genre_finance["est_profit"],
    marker_color="#e07b39",
    hovertemplate="<b>%{x}</b><br>Est. Profit: £%{y:,.2f}<extra></extra>"
))

fig1.update_layout(
    title=dict(
        text="Ticket Revenue, Bar Revenue and Estimated Profit by Genre",
        font=dict(size=18, color="#1a1a2e"),
        x=0.5
    ),
    barmode="group",
    xaxis_title="Event Genre",
    yaxis_title="£ (GBP)",
    yaxis_tickprefix="£",
    yaxis_tickformat=",.0f",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    plot_bgcolor="#ffffff",
    paper_bgcolor="#f8f9fa",
    font=dict(family="Arial", size=13),
    hovermode="x unified",
    height=500
)

# Zero line for profit reference
fig1.add_hline(y=0, line_dash="dash", line_color="grey", opacity=0.5)

fig1.show()
fig1.write_html("chart1_revenue_by_genre.html")
fig1.write_image("chart1_revenue_by_genre.png", width=1200, height=500, scale=2)
print("Chart 1 saved as interactive HTML and PNG.")


### Visualisation 2 — Bar Spend Per Attendee by Genre

**Justification:** A horizontal bar chart is appropriate here because it makes 
ranking across many categories easy to read at a glance. Colour-coding bars 
above/below the average provides immediate insight without requiring the viewer 
to calculate the mean themselves — this is good data communication practice 
(Tufte, 2001). This chart answers research question 2 by showing which genres 
drive the most secondary (bar) revenue per person.


In [ ]:
# ── Visualisation 2: Bar Spend Per Attendee by Genre (Plotly) ────────────
df_bar = df_q2.sort_values("bar_spend_per_attendee", ascending=True).copy()
mean_spend = df_bar["bar_spend_per_attendee"].mean()

# Colour bars above/below average
bar_colours = [
    "#2e4a7a" if v >= mean_spend else "#c0392b"
    for v in df_bar["bar_spend_per_attendee"]
]

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    x=df_bar["bar_spend_per_attendee"],
    y=df_bar["genre"],
    orientation="h",
    marker_color=bar_colours,
    text=[f"£{v:.2f}" for v in df_bar["bar_spend_per_attendee"]],
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Bar spend/attendee: £%{x:.2f}<br>"
        "Bar orders: %{customdata[0]}<br>"
        "Total bar revenue: £%{customdata[1]:,.2f}"
        "<extra></extra>"
    ),
    customdata=df_bar[["bar_orders","total_bar_revenue"]].values
))

# Average line
fig2.add_vline(
    x=mean_spend,
    line_dash="dash",
    line_color="#555555",
    annotation_text=f"Average: £{mean_spend:.2f}",
    annotation_position="top right",
    annotation_font_size=12
)

fig2.update_layout(
    title=dict(
        text="Average Bar Spend Per Attendee by Genre",
        font=dict(size=18, color="#1a1a2e"),
        x=0.5
    ),
    xaxis_title="Average Spend Per Attendee (£)",
    xaxis_tickprefix="£",
    xaxis_tickformat=",.2f",
    yaxis_title="Genre",
    plot_bgcolor="#ffffff",
    paper_bgcolor="#f8f9fa",
    font=dict(family="Arial", size=13),
    height=480,
    margin=dict(l=20, r=120, t=80, b=60),
    showlegend=False
)

fig2.show()
fig2.write_html("chart2_bar_spend_per_attendee.html")
fig2.write_image("chart2_bar_spend_per_attendee.png", width=1100, height=480, scale=2)
print("Chart 2 saved as interactive HTML and PNG.")


### Visualisation 3 — Discount Usage and Attendance Rate by Customer Age Band

**Justification:** A dual-axis chart combining bars and a line is appropriate 
when comparing two related but differently-scaled metrics across the same 
categories. The bars show promotional uptake (a volume measure) while the 
line shows attendance rate (a percentage), allowing the venue to spot where 
discounts are being used but not converting to physical attendance — a key 
operational insight for marketing strategy (research question 3).


In [ ]:
# ── Visualisation 3: Discount Usage vs Attendance Rate (Plotly) ──────────
df_seg = df_q3.sort_values("age_band").copy()

fig3 = make_subplots(specs=[[{"secondary_y": True}]])

# Bars — discount usage %
fig3.add_trace(
    go.Bar(
        name="Discount Usage (%)",
        x=df_seg["age_band"],
        y=df_seg["discount_usage_pct"],
        marker_color="#2e6e6e",
        opacity=0.85,
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Discount usage: %{y:.1f}%<br>"
            "Discounted tickets: %{customdata}<extra></extra>"
        ),
        customdata=df_seg["discounted_tickets"]
    ),
    secondary_y=False
)

# Line — attendance rate %
fig3.add_trace(
    go.Scatter(
        name="Attendance Rate (%)",
        x=df_seg["age_band"],
        y=df_seg["attendance_rate_pct"],
        mode="lines+markers",
        line=dict(color="#c0392b", width=3),
        marker=dict(size=9, color="#c0392b"),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Attendance rate: %{y:.1f}%<br>"
            "Attended: %{customdata}<extra></extra>"
        ),
        customdata=df_seg["attended"]
    ),
    secondary_y=True
)

fig3.update_layout(
    title=dict(
        text="Discount Usage and Attendance Rate by Customer Age Band",
        font=dict(size=18, color="#1a1a2e"),
        x=0.5
    ),
    xaxis_title="Customer Age Band",
    plot_bgcolor="#ffffff",
    paper_bgcolor="#f8f9fa",
    font=dict(family="Arial", size=13),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
    height=500,
    bargap=0.35
)

fig3.update_yaxes(
    title_text="Discount Usage (%)",
    ticksuffix="%",
    secondary_y=False,
    color="#2e6e6e"
)
fig3.update_yaxes(
    title_text="Attendance Rate (%)",
    ticksuffix="%",
    range=[0, 110],
    secondary_y=True,
    color="#c0392b"
)

fig3.show()
fig3.write_html("chart3_segment_analysis.html")
fig3.write_image("chart3_segment_analysis.png", width=1100, height=500, scale=2)
print("Chart 3 saved as interactive HTML and PNG.")


## Step 8 — Download all output files


In [ ]:
# Download everything to your computer
from google.colab import files

download_files = [
    "GrassrootsVenue.db",
    "chart1_revenue_by_genre.html",
    "chart2_bar_spend_per_attendee.html",
    "chart3_segment_analysis.html",
    "chart1_revenue_by_genre.png",
    "chart2_bar_spend_per_attendee.png",
    "chart3_segment_analysis.png",
]

for fname in download_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f"  Downloaded: {fname}")
    else:
        print(f"  NOT FOUND (run earlier cells first): {fname}")


In [ ]:
conn.close()
print("Database connection closed.")
print("\nWhat to do next:")
print("  1. Share this notebook: File → Share → Anyone with the link")
print("  2. Copy the link into your report as the publicly accessible dashboard URL")
print("  3. Insert the PNG charts into Section 5 of your report")
print("  4. Screenshot the SQL query outputs for evidence in Section 5")
